# ⚡ Projeto G1 — Análise da Produção de Energia no Brasil

## Disciplina: Linguagem de Programação — Análise e Visualização de Dados com Python

**Professor:** Alexandre Neves Louzada  
**Aluno:** Ygor Barbosa Gomes

Este notebook apresenta a etapa de análise exploratória e preparação dos dados do projeto sobre a produção de energia no Brasil.

O objetivo é analisar a evolução da produção energética brasileira entre 2015 e 2024, considerando diferentes fontes de energia, regiões, estados, consumo, participação de fontes renováveis e impactos ambientais.

Os resultados obtidos serão utilizados posteriormente na construção de um dashboard interativo desenvolvido com Streamlit.

# 2. Contextualização Energética

A produção de energia é um elemento fundamental para o desenvolvimento econômico e para o funcionamento da infraestrutura de um país.

O Brasil possui uma matriz energética diversificada, formada por diferentes fontes de geração, como hidrelétrica, termelétrica, solar, eólica, biomassa e nuclear.

A análise da produção energética permite compreender a participação dessas fontes na matriz brasileira, identificar diferenças entre regiões e estados, acompanhar a evolução das fontes renováveis e avaliar aspectos relacionados ao consumo e aos impactos ambientais.

Neste projeto, serão analisados dados referentes ao período de 2015 a 2024.

# 3. Explicação da Base de Dados

A base de dados utilizada no projeto contém informações simuladas sobre a produção e o consumo de energia no Brasil entre os anos de 2015 e 2024.

As principais informações disponíveis são:

- **ano:** ano da medição;
- **mes:** mês da medição;
- **data:** data de referência;
- **regiao:** região brasileira;
- **uf:** unidade federativa;
- **fonte_energia:** fonte utilizada para geração de energia;
- **producao_mwh:** quantidade de energia produzida;
- **consumo_mwh:** quantidade de energia consumida;
- **capacidade_instalada:** capacidade de geração instalada;
- **emissao_co2:** emissão estimada de CO₂;
- **custo_medio_mwh:** custo médio da energia;
- **percentual_renovavel:** percentual de energia renovável;
- **nivel_demanda:** classificação do nível de demanda.

# 4. Leitura dos Dados

Nesta etapa, serão importadas as bibliotecas necessárias e realizada a leitura da base de dados utilizada no projeto.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [5]:
df = pd.read_csv("simulacao_producao_energia_brasil.csv")

df.head()

,ano,mes,data,regiao,uf,fonte_energia,producao_mwh,consumo_mwh,capacidade_instalada,emissao_co2,custo_medio_mwh,percentual_renovavel,nivel_demanda
0,2015,1,2015-01-01,Norte,AM,Hidrelétrica,80517.98,88545.10,112725.18,3220.72,190.27,95,Alto
1,2015,1,2015-01-01,Norte,AM,Eólica,30360.65,33758.78,42504.91,1214.43,220.31,95,Alto
2,2015,1,2015-01-01,Norte,AM,Solar,16483.09,15341.74,23076.32,659.32,409.55,95,Médio
3,2015,1,2015-01-01,Norte,AM,Termelétrica,37984.27,41240.83,53177.98,24689.78,346.92,10,Alto
4,2015,1,2015-01-01,Norte,AM,Biomassa,22243.62,24886.82,31141.07,889.74,286.56,95,Alto


# 5. Limpeza e Preparação dos Dados

Nesta etapa, será realizado um check-up inicial da base para verificar suas dimensões, tipos de dados, valores ausentes e registros duplicados.

Em seguida, os dados serão preparados para as análises posteriores.

In [6]:
print("Dimensões da base:", df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nRegistros duplicados:", df.duplicated().sum())

Dimensões da base: (14400, 13)

Tipos de dados:
ano                       int64
mes                       int64
data                     object
regiao                   object
uf                       object
fonte_energia            object
producao_mwh            float64
consumo_mwh             float64
capacidade_instalada    float64
emissao_co2             float64
custo_medio_mwh         float64
percentual_renovavel      int64
nivel_demanda            object
dtype: object

Valores nulos:
ano                     0
mes                     0
data                    0
regiao                  0
uf                      0
fonte_energia           0
producao_mwh            0
consumo_mwh             0
capacidade_instalada    0
emissao_co2             0
custo_medio_mwh         0
percentual_renovavel    0
nivel_demanda           0
dtype: int64

Registros duplicados: 0


In [7]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")

df.head()

,ano,mes,data,regiao,uf,fonte_energia,producao_mwh,consumo_mwh,capacidade_instalada,emissao_co2,custo_medio_mwh,percentual_renovavel,nivel_demanda
0,2015,1,2015-01-01,Norte,AM,Hidrelétrica,80517.98,88545.10,112725.18,3220.72,190.27,95,Alto
1,2015,1,2015-01-01,Norte,AM,Eólica,30360.65,33758.78,42504.91,1214.43,220.31,95,Alto
2,2015,1,2015-01-01,Norte,AM,Solar,16483.09,15341.74,23076.32,659.32,409.55,95,Médio
3,2015,1,2015-01-01,Norte,AM,Termelétrica,37984.27,41240.83,53177.98,24689.78,346.92,10,Alto
4,2015,1,2015-01-01,Norte,AM,Biomassa,22243.62,24886.82,31141.07,889.74,286.56,95,Alto


# 6. Engenharia de Atributos

Nesta etapa, serão criadas novas variáveis a partir dos dados existentes para facilitar as análises temporais e a comparação entre as fontes de energia.

Também será realizada a classificação das fontes entre renováveis e não renováveis.

In [9]:
# Criação de atributos temporais
df["ano_mes"] = df["data"].dt.to_period("M").astype(str)
df["trimestre"] = df["data"].dt.quarter

# Classificação das fontes de energia
fontes_renovaveis = [
    "Hidrelétrica",
    "Eólica",
    "Solar",
    "Biomassa"
]

df["tipo_energia"] = np.where(
    df["fonte_energia"].isin(fontes_renovaveis),
    "Renovável",
    "Não renovável"
)

df[["data", "ano_mes", "trimestre", "fonte_energia", "tipo_energia"]].head()

,data,ano_mes,trimestre,fonte_energia,tipo_energia
0,2015-01-01,2015-01,1,Hidrelétrica,Renovável
1,2015-01-01,2015-01,1,Eólica,Renovável
2,2015-01-01,2015-01,1,Solar,Renovável
3,2015-01-01,2015-01,1,Termelétrica,Não renovável
4,2015-01-01,2015-01,1,Biomassa,Renovável


# 7. KPIs Principais

Nesta etapa, serão calculados os principais indicadores relacionados à produção de energia no Brasil.

Os KPIs analisados são:

- Produção total de energia;
- Fonte energética predominante;
- Percentual médio de energia renovável;
- Estado com maior produção;
- Consumo total de energia;
- Emissão total estimada de CO₂.

In [10]:
# Produção total de energia
producao_total = df["producao_mwh"].sum()

# Fonte com maior produção
producao_por_fonte = (
    df.groupby("fonte_energia")["producao_mwh"]
    .sum()
    .sort_values(ascending=False)
)

fonte_predominante = producao_por_fonte.index[0]

# Percentual renovável médio
percentual_renovavel_medio = df["percentual_renovavel"].mean()

# Estado com maior produção
producao_por_uf = (
    df.groupby("uf")["producao_mwh"]
    .sum()
    .sort_values(ascending=False)
)

estado_maior_producao = producao_por_uf.index[0]

# Consumo total
consumo_total = df["consumo_mwh"].sum()

# Emissão total de CO₂
emissao_total = df["emissao_co2"].sum()

print(f"Produção total: {producao_total:,.2f} MWh")
print(f"Fonte predominante: {fonte_predominante}")
print(f"Percentual renovável médio: {percentual_renovavel_medio:.2f}%")
print(f"Estado com maior produção: {estado_maior_producao}")
print(f"Consumo total: {consumo_total:,.2f} MWh")
print(f"Emissão total de CO₂: {emissao_total:,.2f}")

Produção total: 571,826,344.60 MWh
Fonte predominante: Hidrelétrica
Percentual renovável médio: 80.83%
Estado com maior produção: ES
Consumo total: 543,010,145.33 MWh
Emissão total de CO₂: 87,291,659.65


# 8. Visualizações

## 8.1 Evolução da Produção de Energia

O gráfico a seguir apresenta a evolução da produção total de energia no Brasil ao longo dos anos analisados.

In [11]:
producao_anual = (
    df.groupby("ano")["producao_mwh"]
    .sum()
    .reset_index()
)

fig = px.line(
    producao_anual,
    x="ano",
    y="producao_mwh",
    markers=True,
    title="Evolução da Produção de Energia no Brasil"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="Produção (MWh)"
)

fig.show()

## 8.2 Comparação entre Fontes Energéticas

O gráfico a seguir compara a produção total de energia entre as diferentes fontes presentes na matriz energética brasileira.
\

In [12]:
producao_fonte = (
    df.groupby("fonte_energia")["producao_mwh"]
    .sum()
    .reset_index()
    .sort_values("producao_mwh", ascending=False)
)

fig = px.bar(
    producao_fonte,
    x="fonte_energia",
    y="producao_mwh",
    title="Produção de Energia por Fonte"
)

fig.update_layout(
    xaxis_title="Fonte de Energia",
    yaxis_title="Produção (MWh)"
)

fig.show()

## 8.3 Participação das Fontes na Matriz Energética

O gráfico a seguir apresenta a participação de cada fonte de energia na produção total, permitindo visualizar a composição da matriz energética analisada.

In [13]:
fig = px.pie(
    producao_fonte,
    names="fonte_energia",
    values="producao_mwh",
    title="Participação das Fontes na Matriz Energética",
    hole=0.3
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label"
)

fig.show()

## 8.4 Comparação da Produção por Região

O gráfico a seguir compara a produção total de energia entre as regiões brasileiras, permitindo identificar quais regiões apresentam os maiores volumes de geração.

In [14]:
producao_regiao = (
    df.groupby("regiao")["producao_mwh"]
    .sum()
    .reset_index()
    .sort_values("producao_mwh", ascending=False)
)

fig = px.bar(
    producao_regiao,
    x="regiao",
    y="producao_mwh",
    title="Produção de Energia por Região"
)

fig.update_layout(
    xaxis_title="Região",
    yaxis_title="Produção (MWh)"
)

fig.show()

## 8.5 Ranking de Produção por Estado

O gráfico a seguir apresenta o ranking das UFs de acordo com a produção total de energia, permitindo identificar os estados com maior geração no período analisado.

In [15]:
producao_uf = (
    df.groupby("uf")["producao_mwh"]
    .sum()
    .reset_index()
    .sort_values("producao_mwh", ascending=False)
)

fig = px.bar(
    producao_uf,
    x="producao_mwh",
    y="uf",
    orientation="h",
    title="Ranking de Produção de Energia por UF"
)

fig.update_layout(
    xaxis_title="Produção (MWh)",
    yaxis_title="UF",
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

## 8.6 Evolução da Energia Renovável

O gráfico a seguir apresenta a evolução do percentual médio de energia renovável ao longo dos anos, permitindo observar possíveis mudanças na participação das fontes renováveis na matriz energética.

In [16]:
renovavel_anual = (
    df.groupby("ano")["percentual_renovavel"]
    .mean()
    .reset_index()
)

fig = px.line(
    renovavel_anual,
    x="ano",
    y="percentual_renovavel",
    markers=True,
    title="Evolução do Percentual de Energia Renovável"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="Percentual Renovável (%)"
)

fig.show()

## 8.7 Relação entre Produção e Consumo

O gráfico a seguir analisa a relação entre a quantidade de energia produzida e a quantidade consumida, permitindo observar o comportamento dessas duas variáveis.

In [17]:
fig = px.scatter(
    df,
    x="producao_mwh",
    y="consumo_mwh",
    title="Relação entre Produção e Consumo",
    opacity=0.6
)

fig.update_layout(
    xaxis_title="Produção (MWh)",
    yaxis_title="Consumo (MWh)"
)

fig.show()

In [18]:
correlacao_producao_consumo = df[
    ["producao_mwh", "consumo_mwh"]
].corr().iloc[0, 1]

print(
    f"Correlação entre produção e consumo: "
    f"{correlacao_producao_consumo:.3f}"
)

Correlação entre produção e consumo: 0.977


## 8.8 Impacto Ambiental — Emissão de CO₂ × Produção

O gráfico a seguir analisa a relação entre a produção de energia e a emissão estimada de CO₂, permitindo observar o impacto ambiental associado aos diferentes níveis de produção energética.

In [19]:
fig = px.scatter(
    df,
    x="producao_mwh",
    y="emissao_co2",
    color="fonte_energia",
    title="Emissão de CO₂ × Produção de Energia",
    opacity=0.7
)

fig.update_layout(
    xaxis_title="Produção (MWh)",
    yaxis_title="Emissão de CO₂"
)

fig.show()

## 8.9 Sazonalidade da Produção de Energia

A análise de sazonalidade permite observar como a produção de energia varia entre os meses do ano, identificando possíveis períodos de maior ou menor geração.

In [20]:
producao_mensal = (
    df.groupby("mes")["producao_mwh"]
    .mean()
    .reset_index()
)

fig = px.line(
    producao_mensal,
    x="mes",
    y="producao_mwh",
    markers=True,
    title="Produção Média de Energia por Mês"
)

fig.update_layout(
    xaxis_title="Mês",
    yaxis_title="Produção Média (MWh)",
    xaxis=dict(dtick=1)
)

fig.show()

## 8.10 Evolução da Participação Hidrelétrica

A análise a seguir apresenta a participação da geração hidrelétrica na produção total de energia ao longo dos anos, permitindo observar se houve redução ou aumento da dependência dessa fonte na matriz energética.

In [21]:
producao_total_ano = (
    df.groupby("ano")["producao_mwh"]
    .sum()
)

producao_hidreletrica_ano = (
    df[df["fonte_energia"] == "Hidrelétrica"]
    .groupby("ano")["producao_mwh"]
    .sum()
)

participacao_hidreletrica = (
    producao_hidreletrica_ano / producao_total_ano * 100
).reset_index(name="participacao_percentual")

fig = px.line(
    participacao_hidreletrica,
    x="ano",
    y="participacao_percentual",
    markers=True,
    title="Participação da Energia Hidrelétrica ao Longo dos Anos"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="Participação Hidrelétrica (%)"
)

fig.show()

# 9. Interpretação dos Resultados

Após a análise exploratória, é possível interpretar os principais padrões relacionados à produção de energia no Brasil.

A análise considera a evolução temporal da produção, a participação das diferentes fontes energéticas, as diferenças regionais e estaduais, a participação das fontes renováveis, a relação entre produção e consumo e os impactos ambientais associados à geração de energia.

In [22]:
print("Fonte predominante:", fonte_predominante)
print("Estado com maior produção:", estado_maior_producao)

regiao_maior = (
    df.groupby("regiao")["producao_mwh"]
    .sum()
    .idxmax()
)

print("Região com maior produção:", regiao_maior)

print(
    "Correlação produção x consumo:",
    round(correlacao_producao_consumo, 3)
)

print(
    "Participação hidrelétrica em 2015:",
    round(
        participacao_hidreletrica.loc[
            participacao_hidreletrica["ano"] == 2015,
            "participacao_percentual"
        ].iloc[0],
        2
    ),
    "%"
)

print(
    "Participação hidrelétrica em 2024:",
    round(
        participacao_hidreletrica.loc[
            participacao_hidreletrica["ano"] == 2024,
            "participacao_percentual"
        ].iloc[0],
        2
    ),
    "%"
)

print(
    "Percentual renovável em 2015:",
    round(
        renovavel_anual.loc[
            renovavel_anual["ano"] == 2015,
            "percentual_renovavel"
        ].iloc[0],
        2
    ),
    "%"
)

print(
    "Percentual renovável em 2024:",
    round(
        renovavel_anual.loc[
            renovavel_anual["ano"] == 2024,
            "percentual_renovavel"
        ].iloc[0],
        2
    ),
    "%"
)

Fonte predominante: Hidrelétrica
Estado com maior produção: ES
Região com maior produção: Nordeste
Correlação produção x consumo: 0.977
Participação hidrelétrica em 2015: 37.28 %
Participação hidrelétrica em 2024: 36.9 %
Percentual renovável em 2015: 80.83 %
Percentual renovável em 2024: 80.83 %


## Interpretação dos Resultados

A análise dos dados mostra que a **Hidrelétrica** é a fonte predominante na produção de energia da base analisada.

Entre as regiões brasileiras, o **Nordeste** apresenta a maior produção acumulada de energia, enquanto o **Espírito Santo (ES)** aparece como a UF com maior produção no período analisado.

A relação entre produção e consumo apresentou uma correlação de **0,977**, indicando uma forte associação linear positiva entre as duas variáveis. Isso significa que, na base analisada, maiores níveis de produção estão fortemente associados a maiores níveis de consumo.

Em relação às fontes renováveis, o percentual médio foi de **80,83% em 2015** e permaneceu em **80,83% em 2024**. Portanto, considerando esses dois anos, não foi observado crescimento no percentual renovável.

A participação da energia hidrelétrica passou de **37,28% em 2015 para 36,90% em 2024**, representando uma pequena redução de **0,38 ponto percentual**. Dessa forma, os dados indicam uma leve redução da participação hidrelétrica, mas não uma mudança expressiva na composição da matriz energética.

Os resultados mostram que a matriz analisada possui forte participação de fontes renováveis, ao mesmo tempo em que a geração hidrelétrica continua sendo a principal fonte individual de produção.

# 10. Conclusão

A análise da produção de energia no Brasil entre 2015 e 2024 permitiu identificar diferenças na geração entre fontes energéticas, regiões e estados, além de analisar aspectos relacionados ao consumo, à participação de fontes renováveis e aos impactos ambientais.

A fonte hidrelétrica apresentou a maior produção acumulada na base analisada, enquanto a região Nordeste se destacou pela maior produção regional e o Espírito Santo (ES) apresentou a maior produção entre as UFs.

A análise também mostrou uma forte associação positiva entre produção e consumo de energia. Em relação à matriz energética, o percentual renovável permaneceu estável entre 2015 e 2024, enquanto a participação hidrelétrica apresentou uma pequena redução no período.

As visualizações desenvolvidas com Plotly facilitaram a comparação e a interpretação dos dados, permitindo uma análise mais clara da evolução e da composição da produção energética.

Os resultados obtidos servirão como base para o desenvolvimento de um dashboard interativo em Streamlit, no qual será possível explorar os indicadores e visualizações através de diferentes filtros.

# Persistência dos Dados com SQLAlchemy e SQLite

Como funcionalidade adicional, os dados analisados serão armazenados em um banco de dados SQLite utilizando SQLAlchemy.

Essa etapa permite demonstrar a persistência dos dados e a realização de consultas SQL diretamente sobre a base armazenada.

In [23]:
engine = create_engine("sqlite:///producao_energia.sqlite")

df.to_sql(
    "producao_energia",
    engine,
    if_exists="replace",
    index=False
)

print("Tabela producao_energia criada no banco SQLite.")

Tabela producao_energia criada no banco SQLite.


In [24]:
consulta_regiao = pd.read_sql(
    """
    SELECT
        regiao,
        SUM(producao_mwh) AS producao_total
    FROM producao_energia
    GROUP BY regiao
    ORDER BY producao_total DESC
    """,
    engine
)

consulta_regiao

,regiao,producao_total
0,Nordeste,1.433834e+08
1,Sudeste,1.147399e+08
2,Centro-Oeste,1.142929e+08
3,Norte,1.140422e+08
4,Sul,8.536788e+07
